In [13]:
import os
import tiktoken
import json
from dotenv import load_dotenv
from datasets import load_dataset, Dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from huggingface_hub import login
from openai import OpenAI

"""imports related to Embeddings and Vector Stores"""
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [2]:
load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')

if openai_key:
    print(f'OpenAi key exists and starts with {openai_key[:8]}')
else:
    print('OpenAi key does not exists')

OpenAi key exists and starts with sk-proj-


In [3]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


In [4]:
dataset = load_dataset("Pratheep17/rag-mini-wikipedia-refined", split="train")

if dataset:
    print(f"Total Number of text-corpus: {len(dataset)}")
else:
    print('Issue occured while loading the datasets')

Total Number of text-corpus: 3198


In [5]:
dataset = dataset.select_columns(['id', 'passage'])

In [6]:
MODEL="gpt-4.1-nano"
db_name="vector_db"

In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings, collection_name=db_name)

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for item in dataset:
    text = item["passage"]

    if not text or not text.strip():
        continue

    if len(text) <= 1200:
        chunks.append({
            "id": item["id"],
            "chunk_id": f"{item['id']}_0",
            "text": text,
            "metadata": {
                "source_id": item["id"],
                "chunk_id": f"{item['id']}_0"
            }
        })
    else:
        split_texts = splitter.split_text(text)

        for i, chunk in enumerate(split_texts):
            chunks.append({
                "id": item["id"],
                "chunk_id": f"{item['id']}_{i}",
                "text": chunk,
                "metadata": {
                    "source_id": item["id"],
                    "chunk_index": i,
                    "chunk_id": f"{item['id']}_{i}"
                }
            })

In [9]:
if os.path.exists(db_name):
    print(f"Loading existing vector store from {db_name}...")
    Chroma(collection_name=db_name, embedding_function=embeddings, persist_directory=db_name).delete_collection()

vector_store = Chroma.from_texts(
    texts=[chunk["text"] for chunk in chunks],
    ids=[chunk["chunk_id"] for chunk in chunks],
    metadatas=[chunk.get("metadata", {}) for chunk in chunks],
    embedding=embeddings,
    collection_name=db_name,
    persist_directory=db_name
)
print(f"Vector store created with {vector_store._collection.count()} vectors.")

Loading existing vector store from vector_db...
Vector store created with 3298 vectors.


In [19]:
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import time
from litellm import completion

@dataclass
class RAGConfig:
    """Configuration for the RAG pipeline"""
    embedding_model: str = "text-embedding-3-small"
    llm_model: str = "gpt-4.1-nano"
    reranker_model: str = "gpt-5-mini"
    chunk_size: int = 1000
    chunk_overlap: int = 150
    retrieval_k: int = 15
    rerank_top_n: int = 5
    temperature: float = 0.0
    persist_directory: str = "vector_db"
    collection_name: str = "vector_db"
    conversation_window: int = 6  # Number of recent turns to consider

class ProductionRAGPipeline:
    """
    Industry-standard RAG pipeline with retrieval, reranking, and generation.
    
    Features:
    - Semantic search with ChromaDB
    - LLM-based reranking for improved relevance
    - Context-aware response generation
    - Performance metrics and logging
    """
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self._initialize_components()
    
    def _initialize_components(self):
        """Initialize embeddings, vector store, and LLMs"""
        print("Initializing RAG pipeline components...")
        
        # Embeddings
        self.embeddings = OpenAIEmbeddings(model=self.config.embedding_model)
        
        # Vector store
        self.vector_store = Chroma(
            persist_directory=self.config.persist_directory,
            embedding_function=self.embeddings,
            collection_name=self.config.collection_name
        )
        
        # LLMs
        self.llm = ChatOpenAI(
            model=self.config.llm_model,
            temperature=self.config.temperature
        )
        
        self.reranker_llm = ChatOpenAI(
            model=self.config.reranker_model,
            temperature=self.config.temperature
        )
        
        print(f"✓ Loaded vector store with {self.vector_store._collection.count()} vectors")
        print(f"✓ Initialized {self.config.llm_model} for generation")
        print(f"✓ Initialized {self.config.reranker_model} for reranking")
    
    def _format_history(self, history: List[Dict[str, str]]) -> str:
        """Format conversation history for context"""
        recent_turns = history[-self.config.conversation_window :]
        if not recent_turns:
            return "No previous conversation."
        
        lines = []
        for turn in recent_turns:
            role = turn.get("role", "unknown")
            content = turn.get("content", "")
            lines.append(f"{role}: {content}")
        return "\n".join(lines)
    
    def rewrite_question(self, question: str, history: Optional[List[Dict[str, str]]] = None) -> str:
        """
        Rewrite user question based on conversation history to create a standalone query.
        
        Args:
            question: Current user question
            history: List of conversation turns with 'role' and 'content' keys
            
        Returns:
            Rewritten standalone question
        """
        if not history:
            return question
        
        prompt = f"""Given the conversation history and the latest user question, rewrite the latest
question as a standalone retrieval query.

Rules:
- Resolve pronouns and references such as he, she, his, her, it, they, that place, and the former.
- Preserve the user's original intent.
- Do not answer the question.
- Do not add facts that are not implied by the conversation.
- Return only the standalone question.

Conversation history:
{self._format_history(history)}

Latest user question:
{question}"""
        
        response = self.llm.invoke(prompt)
        rewritten = response.content.strip().strip('"')
        return rewritten or question
    
    def retrieve(self, query: str, k: Optional[int] = None) -> List[Document]:
        """
        Stage 1: Retrieve relevant documents using semantic search
        
        Args:
            query: User question
            k: Number of documents to retrieve
            
        Returns:
            List of Document objects
        """
        k = k or self.config.retrieval_k
        
        # Embed the query
        query_embedding = self.embeddings.embed_query(query)
        
        # Search vector store
        results = self.vector_store._collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )
        
        # Convert to Documents
        docs = []
        for doc_text, metadata in zip(results["documents"][0], results["metadatas"][0]):
            docs.append(Document(page_content=doc_text, metadata=metadata))
        
        return docs
    
    def rerank(self, query: str, docs: List[Document], top_n: Optional[int] = None) -> List[Document]:
        """
        Stage 2: Rerank documents using LLM for improved relevance
        
        Args:
            query: User question
            docs: Retrieved documents
            top_n: Number of top documents to return
            
        Returns:
            Reranked list of Document objects
        """
        top_n = top_n or self.config.rerank_top_n
        
        if not docs:
            return []
        
        # Prepare candidates
        candidates = []
        for i, doc in enumerate(docs):
            candidates.append({
                "index": i,
                "source_id": doc.metadata.get("source_id", "N/A"),
                "chunk_id": doc.metadata.get("chunk_id", "N/A"),
                "passage": doc.page_content
            })
        
        # Rerank prompt
        prompt = f"""You are a precise reranking model for a Wikipedia-style RAG system.

Given a user query and candidate passages, rank the passages by how useful they are for answering the query.

Rules:
1. Prefer passages that directly answer the query.
2. Prefer factual specificity over broad relatedness.
3. Penalize passages that only share keywords but do not answer the query.
4. Return only valid JSON.
5. Do not explain your reasoning.

Return JSON in this format:
{{
  "ranked_results": [
    {{"index": 0, "score": 0.95}},
    {{"index": 2, "score": 0.82}}
  ]
}}

User query:
{query}

Candidate passages:
{json.dumps(candidates, ensure_ascii=False, indent=2)}
"""
        
        # Call reranker
        response = self.reranker_llm.invoke(prompt)
        
        try:
            result = json.loads(response.content)
            ranked_results = result["ranked_results"]
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Reranking failed: {e}. Returning original order.")
            return docs[:top_n]
        
        # Build reranked documents
        reranked_docs = []
        for item in ranked_results[:top_n]:
            doc_index = item["index"]
            if 0 <= doc_index < len(docs):
                doc = docs[doc_index]
                doc.metadata["rerank_score"] = item.get("score", 0.0)
                reranked_docs.append(doc)
        
        return reranked_docs
    
    def _build_answer_prompt(
        self,
        query: str,
        context_docs: List[Document],
        standalone_question: Optional[str] = None,
    ) -> str:
        """Build the final grounded answer prompt used by normal and streaming generation."""
        context = "\n\n".join(
            f"source_id={d.metadata.get('source_id')} "
            f"chunk_id={d.metadata.get('chunk_id')}\n{d.page_content}"
            for d in context_docs
        )

        if standalone_question and standalone_question != query:
            question_block = (
                f"Original question: {query}\n"
                f"Standalone question resolved from conversation: {standalone_question}\n"
                "Answer the standalone question."
            )
        else:
            question_block = f"Question: {query}"

        return (
            "You are a grounded Wikipedia-style RAG assistant. "
            "Use only the context below to answer. "
            "If the answer is not in the context, say you do not have enough information.\n\n"
            f"Context:\n{context}\n\n"
            f"{question_block}"
        )

    def generate(self, query: str, context_docs: List[Document], standalone_question: Optional[str] = None) -> str:
        """
        Stage 3: Generate a complete non-streaming answer.
        """
        if not context_docs:
            return "No relevant context was found to answer your question."

        prompt = self._build_answer_prompt(query, context_docs, standalone_question)
        response = self.llm.invoke(prompt)
        return response.content

    @staticmethod
    def _stream_delta_content(chunk) -> str:
        """Extract text from LiteLLM/OpenAI-compatible streaming chunks."""
        try:
            return chunk["choices"][0]["delta"].get("content") or ""
        except (KeyError, IndexError, TypeError):
            pass

        try:
            delta = chunk.choices[0].delta
            return getattr(delta, "content", None) or ""
        except (AttributeError, IndexError, TypeError):
            return ""

    def generate_stream(
        self,
        query: str,
        context_docs: List[Document],
        standalone_question: Optional[str] = None,
    ):
        """
        Stage 3: Stream the final answer with LiteLLM.

        Retrieval and reranking remain synchronous; only the final answer tokens stream.
        """
        if not context_docs:
            yield "No relevant context was found to answer your question."
            return

        prompt = self._build_answer_prompt(query, context_docs, standalone_question)
        stream = completion(
            model=f"openai/{self.config.llm_model}",
            messages=[{"role": "user", "content": prompt}],
            temperature=self.config.temperature,
            stream=True,
        )

        for chunk in stream:
            token = self._stream_delta_content(chunk)
            if token:
                yield token

    def query(
        self,
        question: str,
        history: Optional[List[Dict[str, str]]] = None,
        retrieval_k: Optional[int] = None,
        rerank_top_n: Optional[int] = None,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        End-to-end RAG pipeline: retrieve, rerank, generate with conversation memory
        """
        start_time = time.time()

        rewrite_start = time.time()
        standalone_question = self.rewrite_question(question, history)
        rewrite_time = time.time() - rewrite_start

        if verbose:
            print(f"\n{'='*80}")
            print(f"Original Question: {question}")
            if standalone_question != question:
                print(f"Standalone Question: {standalone_question}")
            print(f"{'='*80}\n")

        retrieval_start = time.time()
        retrieved_docs = self.retrieve(standalone_question, k=retrieval_k)
        retrieval_time = time.time() - retrieval_start

        if verbose:
            if standalone_question != question:
                print(f"[Stage 0: Rewriting] Rewrote question ({rewrite_time:.3f}s)")
            print(f"[Stage 1: Retrieval] Retrieved {len(retrieved_docs)} documents ({retrieval_time:.3f}s)")

        rerank_start = time.time()
        reranked_docs = self.rerank(standalone_question, retrieved_docs, top_n=rerank_top_n)
        rerank_time = time.time() - rerank_start

        if verbose:
            print(f"[Stage 2: Reranking] Reranked to top {len(reranked_docs)} documents ({rerank_time:.3f}s)")

        generation_start = time.time()
        answer = self.generate(question, reranked_docs, standalone_question=standalone_question)
        generation_time = time.time() - generation_start

        if verbose:
            print(f"[Stage 3: Generation] Generated answer ({generation_time:.3f}s)")

        total_time = time.time() - start_time

        if verbose:
            print(f"\n{'='*80}")
            print(f"Total pipeline time: {total_time:.3f}s")
            print(f"{'='*80}\n")

        return {
            "question": question,
            "standalone_question": standalone_question,
            "answer": answer,
            "retrieved_docs": retrieved_docs,
            "reranked_docs": reranked_docs,
            "metrics": {
                "rewrite_time": rewrite_time if standalone_question != question else 0.0,
                "retrieval_time": retrieval_time,
                "rerank_time": rerank_time,
                "generation_time": generation_time,
                "total_time": total_time,
                "num_retrieved": len(retrieved_docs),
                "num_reranked": len(reranked_docs)
            }
        }

    def query_stream(
        self,
        question: str,
        history: Optional[List[Dict[str, str]]] = None,
        retrieval_k: Optional[int] = None,
        rerank_top_n: Optional[int] = None,
        verbose: bool = False,
    ):
        """
        End-to-end RAG pipeline with streaming final generation.

        Yields dictionaries so callers can display status, partial answer text,
        and debug metadata such as the rewritten standalone question.
        """
        start_time = time.time()

        standalone_question = self.rewrite_question(question, history)
        if verbose and standalone_question != question:
            print(f"Standalone Question: {standalone_question}")

        yield {
            "event": "status",
            "answer": "Retrieving relevant context...",
            "standalone_question": standalone_question,
        }

        retrieval_start = time.time()
        retrieved_docs = self.retrieve(standalone_question, k=retrieval_k)
        retrieval_time = time.time() - retrieval_start

        yield {
            "event": "status",
            "answer": "Reranking retrieved context...",
            "standalone_question": standalone_question,
        }

        rerank_start = time.time()
        reranked_docs = self.rerank(standalone_question, retrieved_docs, top_n=rerank_top_n)
        rerank_time = time.time() - rerank_start

        accumulated = ""
        for token in self.generate_stream(question, reranked_docs, standalone_question=standalone_question):
            accumulated += token
            yield {
                "event": "token",
                "question": question,
                "standalone_question": standalone_question,
                "answer": accumulated,
                "retrieved_docs": retrieved_docs,
                "reranked_docs": reranked_docs,
                "metrics": {
                    "retrieval_time": retrieval_time,
                    "rerank_time": rerank_time,
                    "total_time": time.time() - start_time,
                    "num_retrieved": len(retrieved_docs),
                    "num_reranked": len(reranked_docs),
                },
            }

        if not accumulated:
            accumulated = "The available context does not provide enough information to answer that."
            yield {
                "event": "token",
                "question": question,
                "standalone_question": standalone_question,
                "answer": accumulated,
                "retrieved_docs": retrieved_docs,
                "reranked_docs": reranked_docs,
                "metrics": {
                    "retrieval_time": retrieval_time,
                    "rerank_time": rerank_time,
                    "total_time": time.time() - start_time,
                    "num_retrieved": len(retrieved_docs),
                    "num_reranked": len(reranked_docs),
                },
            }

    def batch_query(self, questions: List[str], verbose: bool = False) -> List[Dict[str, Any]]:
        """
        Process multiple questions in batch
        
        Args:
            questions: List of questions
            verbose: Print details for each question
            
        Returns:
            List of result dictionaries
        """
        results = []
        for i, question in enumerate(questions, 1):
            print(f"\nProcessing question {i}/{len(questions)}...")
            result = self.query(question, verbose=verbose)
            results.append(result)
        return results

15:21:27 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
15:21:28 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [15]:
# Configure the RAG pipeline
config = RAGConfig(
    embedding_model="text-embedding-3-small",
    llm_model="gpt-4.1-nano",
    reranker_model="gpt-5-mini",
    chunk_size=1000,
    chunk_overlap=150,
    retrieval_k=15,
    rerank_top_n=5,
    temperature=0.0,
    persist_directory="vector_db",
    collection_name="vector_db",
    conversation_window=6  # Keep last 6 conversation turns for context
)

# Initialize the pipeline
rag_pipeline = ProductionRAGPipeline(config)

print("\n✓ RAG Pipeline ready for queries!")

Initializing RAG pipeline components...
✓ Loaded vector store with 3298 vectors
✓ Initialized gpt-4.1-nano for generation
✓ Initialized gpt-5-mini for reranking

✓ RAG Pipeline ready for queries!


In [20]:
# Example 1: Query about Uruguay's agriculture
result = rag_pipeline.query(
    question="How much does the agriculture sector contribute to Uruguay's GDP?",
    verbose=True
)

print("\nANSWER:")
print(result["answer"])


Original Question: How much does the agriculture sector contribute to Uruguay's GDP?

[Stage 1: Retrieval] Retrieved 15 documents (1.762s)
[Stage 2: Reranking] Reranked to top 2 documents (6.249s)
[Stage 3: Generation] Generated answer (1.323s)

Total pipeline time: 9.334s


ANSWER:
The agriculture sector contributes roughly 10% to Uruguay's GDP.


In [21]:
import gradio as gr


def normalize_gradio_history(history):
    """
    Convert Gradio history into:
    [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]

    Supports both common Gradio formats:
    1. [{"role": "user", "content": "..."}]
    2. [[user_message, assistant_message]]
    """
    conversation = []

    for turn in history or []:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content")

            if role and content:
                conversation.append({
                    "role": str(role),
                    "content": str(content),
                })

        elif isinstance(turn, (list, tuple)) and len(turn) >= 2:
            user_message, assistant_message = turn[0], turn[1]

            if user_message:
                conversation.append({
                    "role": "user",
                    "content": str(user_message),
                })

            if assistant_message:
                conversation.append({
                    "role": "assistant",
                    "content": str(assistant_message),
                })

    return conversation


def rag_chat_interface(message, history):
    """
    Gradio streaming chat function for conversational RAG.
    """
    if not message or not message.strip():
        yield "Please enter a question."
        return

    try:
        conversation = normalize_gradio_history(history)
        for update in rag_pipeline.query_stream(
            question=message,
            history=conversation or None,
            verbose=False,
        ):
            yield update["answer"]

    except Exception as e:
        yield f"Error: {e}"


demo = gr.ChatInterface(
    fn=rag_chat_interface,
    title="Wikipedia Chat Assistant",
    description=(
        "Ask questions about the Wikipedia knowledge base. "
        "Answers stream after retrieval and reranking complete."
    ),
    examples=[
        "Who was born in Clermont-Ferrand?",
        "When was his date of birth?",
        "What is Uruguay?",
        "How much does agriculture contribute to Uruguay's GDP?",
        "Tell me about the history of Uruguay",
    ],
    chatbot=gr.Chatbot(height=500),
)

print("Streaming chat interface ready!")


Streaming chat interface ready!


In [22]:
# Launch the Gradio interface
# Gradio will automatically find an available port
demo.launch(share=False, show_error=True)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
